## Current issue:

Training sample `AUD0000000538_S0008472` loads an empty wav file, crashing the job

This was due to bad opus 2 wav conversion 

## Fixed on 02/18/2022

In [62]:
import pandas as pd
import kaldiio
from tqdm import tqdm
from pathlib import Path
from os.path import join, getsize
from joblib import Parallel, delayed
import numpy as np 
import torchaudio 

In [2]:
from IPython.display import Audio

In [3]:
path = '/om2/data/public/GigaSpeech/'
bucket_size = 16
split = 'XL'
# Load csv
if type(split) is list:
    split = split[0]
csv_path = Path(path,f"data/{split}.csv")
file_csv = pd.read_csv(csv_path, sep='\t', dtype='str') 
# filter bad files
file_csv = file_csv[~file_csv.isna().any(axis=1)]
# load file segments - csv of excerpt, wav file path, eg start, eg end
segments_path = Path(path, f"data/segments")
segments = pd.read_csv(segments_path, header=None,
               names=['wav_filename', 'speaker', 'start', 'end'],
               sep='\t')
# map of speaker to wav path

# merge file csv with segmnet onsets & offsets
times = segments[['start', 'end']][segments.wav_filename.isin(file_csv.wav_filename)]
file_csv = pd.concat([file_csv, times.set_index(file_csv.index)], axis=1)
# Tokenize transcript
# file_csv['transcript'] = file_csv['transcript'].apply(tokenizer.encode)
# Sort dataset by text length & make attribute
file_csv = file_csv.sort_values(by='transcript', key=lambda x: x.str.len(), ascending=True)


In [4]:
file_csv.head(1)

,wav_filename,wav_len,transcript,speaker,start,end
7413943,AUD0000000304_S0000880,780,O,AUD0000000304,3931.99,3932.77


In [5]:
wavscp = kaldiio.load_scp(Path(path,'data','wav.scp').as_posix())


In [6]:
files = pd.read_csv(Path(path,'data','wav.scp').as_posix(), 
                    sep = '\t', names=['speaker', 'wav_path'],header=None, dtype='str')

In [7]:
files

,speaker,wav_path
0,POD0000000001,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
1,POD0000000002,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
2,POD0000000003,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
3,POD0000000004,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
4,POD0000000005,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
...,...,...
38126,YOU1000000187,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
38127,YOU1000000188,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
38128,YOU1000000189,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...
38129,YOU1000000191,/rdma/vast-rdma/om3/data/public/GigaSpeech/aud...


In [11]:
files[files.speaker=='YOU0000010700'].wav_path.item()

'/rdma/vast-rdma/om3/data/public/GigaSpeech/audio/youtube/P0107/YOU0000010700.wav'

In [ ]:
file_csv.transcript[file_csv.transcript.str.len() < 2].shape

In [ ]:
eg = file_csv.iloc[0]
wav_filename = eg.wav_filename


In [ ]:
eg

In [ ]:
wav = wavscp[eg.speaker][1]
on = int(eg.start * 16000)
off = int(eg.end * 16000)
excerpt = wav[on:off]

In [ ]:
Audio(excerpt, rate=16000)

# Look at bad EG 

In [12]:
eg = file_csv[file_csv.wav_filename == 'YOU0000010700_S0000473']

In [13]:
file_csv[file_csv.speaker == 'YOU0000010700'].sort_values(by='start')

,wav_filename,wav_len,transcript,speaker,start,end
4239686,YOU0000010700_S0000005,3090,THAT POINT WHEN EVERYTHING SEEMS IMPORTANT,YOU0000010700,42.40,45.49
4239687,YOU0000010700_S0000006,1200,AND DIRE,YOU0000010700,45.58,46.78
4239688,YOU0000010700_S0000008,4839,AND YOU KNOW THAT EXCITING POINT IN LIFE IS WH...,YOU0000010700,51.07,55.91
4239689,YOU0000010700_S0000010,809,SO,YOU0000010700,65.90,66.71
4239690,YOU0000010700_S0000011,4709,WHEN I FIRST STARTED THINKING OF WHAT I WANTED...,YOU0000010700,68.09,72.80
...,...,...,...,...,...,...
4240213,YOU0000010700_S0000661,3089,IF I EVER WANTED TO YOU KNOW WRITE SOMETHING B...,YOU0000010700,2995.13,2998.22
4240214,YOU0000010700_S0000662,8019,BUT THEN AGAIN MY STORIES ARE ALWAYS LIKE STAN...,YOU0000010700,2998.37,3006.39
4240215,YOU0000010700_S0000663,4710,YOU KNOW THE READER IS IS LEFT TO ASK THOSE QU...,YOU0000010700,3006.66,3011.37
4240216,YOU0000010700_S0000664,1260,BUT THANK YOU FOR THE QUESTION,YOU0000010700,3012.24,3013.50


In [14]:
eg

,wav_filename,wav_len,transcript,speaker,start,end
4240111,YOU0000010700_S0000473,9370,SO I ENDED UP HAVING TO SEPARATE THE BOOK INTO...,YOU0000010700,1822.62,1831.99


In [15]:
name = eg.wav_filename.item()
speaker = eg.speaker.item()
start = eg.start.item()
end = eg.end.item()
wav = wavscp[speaker][1]
rate = wavscp[speaker][0]

In [22]:
on, off , wav.shape

(29161920, 29311840, (22403824,))

In [16]:
on = int(start * rate )
off = int(end * rate )
Audio(wav[on:off], rate=rate)

ValueError: zero-size array to reduction operation maximum which has no identity

In [ ]:
(len(wav) / 16000)  # time in seconds 

In [24]:
import librosa

In [59]:
librosa.get_duration(filename='/om2/data/public/GigaSpeech/audio/youtube/P0107/YOU0000010700.opus')

3034.5

In [60]:
librosa.get_duration(filename='/om2/data/public/GigaSpeech/audio/youtube/P0107/YOU0000010700.wav')

1400.239

In [65]:
!ffmpeg -i '/om2/data/public/GigaSpeech/audio/youtube/P0107/YOU0000010700.opus' 2>&1 | grep Duration

  Duration: 00:50:34.55, start: 0.000000, bitrate: 32 kb/s


In [52]:
from tqdm import tqdm
import re, os

In [58]:
bad_files = []
wavs = files.wav_path.values
for file in tqdm(wavs):
    wav_time = librosa.get_duration(filename=file)
    opus_pth = Path(file).with_suffix('.opus')
#     print(opus_pth)
    opus_time = librosa.get_duration(filename=opus_pth)
    if wav_time != opus_time:
        bad_files.append(file)
    

  0%|          | 18/38131 [00:14<8:37:50,  1.23it/s]


KeyboardInterrupt: 

In [57]:
Path(file).with_suffix('.opus')

PosixPath('/rdma/vast-rdma/om3/data/public/GigaSpeech/audio/podcast/P0017/POD0000001692.opus')

In [46]:
file

'/rdma/vast-rdma/om3/data/public/GigaSpeech/audio/podcast/P0017/POD0000001692.wav'

In [81]:
with open('/om2/data/public/GigaSpeech/data/opus.scp', 'r') as oscp:
    lines = oscp.read().splitlines()
    for line in lines:
        utt, path = line.split('\t')
        wav_path = path.replace('.opus', '.wav')
        break

In [102]:
len(lines)//100, len(lines)

(381, 38131)

In [104]:
start = (100-1) * 382
end = start + 382 
print(start, end)

37818 38200
